# Research notebook



In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import PICTURE_DIR, GALLERY_DIR, OUTPUT_DIR, LLAVA_MODEL, TARGET_CLASSES

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [1]:
import pickle
with open(OUTPUT_DIR / '中文Clip每类正样本预测结果.pkl', 'rb') as f:
    cPclass=pickle.load(f)
with open(OUTPUT_DIR / '中文Clip每类负样本预测结果.pkl', 'rb') as f:
    cNclass=pickle.load(f)
with open(OUTPUT_DIR / '英文Clip每类正样本预测结果.pkl', 'rb') as f:
    ePclass = pickle.load(f)
with open(OUTPUT_DIR / '英文Clip每类负样本预测结果.pkl', 'rb') as f:
    eNclass = pickle.load(f)

In [2]:
picture = list(TARGET_CLASSES)
if any(set(predictions) != set(picture) for predictions in (ePclass, eNclass, cPclass, cNclass)):
    raise ValueError("Prediction categories differ from TARGET_CLASSES. Regenerate both CLIP outputs with the current configuration.")
print(picture)

['Dog', 'Piano', 'Erhu', 'Porcelain', 'Duck']


In [3]:
Pclass={k:[] for k in picture}
Nclass={k:[] for k in picture}
for i in picture:
    pclass = [int(cPclass[i][x] == 1 or ePclass[i][x] == 1) for x in range(len(cPclass[i]))]
    nclass = [int(cNclass[i][x] == 1 or eNclass[i][x] == 1) for x in range(len(cNclass[i]))]
    Pclass[i]=pclass
    Nclass[i]=nclass

In [4]:
print(Pclass)

{'Dog': [1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1], 'Piano': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [5]:
def evaluation(Pclass,Nclass):
    TP=Pclass.count(1)
    FN=Pclass.count(0)
    FP=Nclass.count(1)
    if TP+FP!=0:
        Precision=TP/(TP+FP)
    if TP+FP==0:
        Precision=0
    if TP+FN!=0:
        Recall=TP/(TP+FN)
    if TP+FN==0:
        Recall=0
    return Precision, Recall

In [8]:
# 不卡阈值
f1_score0 = {k: 0 for k in picture}
for figure, x in zip(picture, range(len(picture))):
    precision, recall = evaluation(Pclass[figure], Nclass[figure])
    if precision + recall == 0:
        f1_score0[figure] = 0
    else:
        f1_score0[figure] = 2 * (precision * recall) / (precision + recall)
        print(f'{figure},precision:{precision:.4f},recall:{recall:.4f},f1:{f1_score0[figure]:.4f}')


Dog,precision:0.8634,recall:0.9800,f1:0.9180
Piano,precision:0.6037,recall:0.9750,f1:0.7457
Erhu,precision:0.8085,recall:0.9500,f1:0.8736
Porcelain,precision:0.6611,recall:0.9950,f1:0.7944
Duck,precision:0.7071,recall:0.9900,f1:0.8250


In [7]:
from pathlib import Path
import os
from PIL import Image
def subfolder(exclude):
    root = PICTURE_DIR
    return [str(item) for subdir in root.iterdir()
            if subdir.is_dir()
            for item in subdir.iterdir()
            if item.is_dir() and item!=(PICTURE_DIR / exclude / exclude)]
def load_images(folder_path):
    images = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            with Image.open(file_path) as img:
                images.append(img.copy())
        except Exception as e:
            print(f"无法打开文件 {filename}: {e}")
    return images
def get_figure_path(p, n):
    Ppath = []
    Npath = []

    # 保存相对于图库根目录的正样本路径，便于迁移数据
    for filename in os.listdir(p):
        file_path = os.path.join(p, filename)
        relative_path = Path(file_path).relative_to(PICTURE_DIR).as_posix()
        Ppath.append(relative_path)

    # 负样本使用相同的相对路径格式
    for a in n:
        for filename in os.listdir(a):
            file_path = os.path.join(a, filename)
            relative_path = Path(file_path).relative_to(PICTURE_DIR).as_posix()
            Npath.append(relative_path)

    return Ppath, Npath

def create_path(figure):
    positive_path=str(PICTURE_DIR / figure / figure)
    negative_path=subfolder(figure)
    return positive_path,negative_path


In [8]:
import pickle
import pandas as pd

In [9]:
for figure in picture:
    positive_path,negative_path=create_path(figure)
    p,n=get_figure_path(positive_path,negative_path)
    path=p+n
    pre=Pclass[figure]+Nclass[figure]
    tru=[1 if i<len(p) else 0 for i in range(len(path))]
    dataset={'Path':path,'True':tru,'Pre':pre}
    with open(OUTPUT_DIR / f'{figure} llava dataset', 'wb') as f:
        pickle.dump(dataset, f)

In [ ]:
print(dataset['Path'])